In [ ]:
pip install plotly

# Cópula empírica

La distribución de datos empíricos puede transformarse en lo que se denomina una "cópula empírica" ​​mediante la deformación, de modo que las distribuciones marginales se vuelvan uniformes. Sean $x, y$ dos muestras de tamaño n cada una. La función de frecuencia de la cópula empírica a menudo puede calcularse para cualquier par $(x,y)$ mediante:\
$C_n(\frac{i}{n},\frac{j}{n})=\frac{\sum_{i=1}^{n}1(x\leq x_{i} and y\leq y_{j})}{n}$

In [ ]:
import pandas as pd
import numpy as np
# Datos de ejemplo (Datos de caudal máximo y volumen de inundación)
data = {
    'Pair': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54],
    'Year': [1945, 1945, 1944, 1945, 1946, 1947, 1948, 1949, 1950, 1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1990, 1970, 1971, 1972, 1974, 1978, 1975, 1977, 1977, 1970, 1979, 1980, 1982, 1982, 1985, 1984, 1985, 1966, 1987, 1988, 1989, 1990, 1991, 1992, 1994, 1994, 1995],
    'V_m3': [8700, 6600, 4419, 8627, 8409, 13002, 5768, 5002, 14157, 10128, 12025, 10328, 8923, 11401, 6500, 5288, 8102, 6414, 8000, 9800, 7335, 8477, 7648, 5906, 8808, 4992, 8692, 11272, 8400, 6969, 9352, 12828, 13608, 5499, 12377, 11427, 9406, 14549, 5657, 9645, 7341, 12553, 13603, 6460, 7502, 8650, 7350, 9568, 6728, 8313, 8041, 10714, 14740, 8711],
    'Q_mm': [311, 325, 189, 230, 249, 331, 376, 191, 184, 235, 286, 233, 230, 351, 385, 168, 343, 214, 300, 300, 141, 222, 122, 121, 186, 173, 292, 436, 244, 246, 237, 271, 442, 260, 236, 334, 340, 382, 351, 197, 283, 305, 405, 176, 181, 203, 187, 216, 196, 245, 255, 257, 232, 266]
}
df = pd.DataFrame(data)
df.head(10)

,Pair,Year,V_m3,Q_mm
0,1,1945,8700,311
1,2,1945,6600,325
2,3,1944,4419,189
3,4,1945,8627,230
4,5,1946,8409,249
5,6,1947,13002,331
6,7,1948,5768,376
7,8,1949,5002,191
8,9,1950,14157,184
9,10,1951,10128,235


In [ ]:
'''
Para determinar la cópula empírica, primero debemos ordenar el volumen de la inundación y
con C_n calculamos las variables de caudal máximo en orden creciente.
'''
def copula_empirica(df, var1, var2):
    n = len(df)

    ## Ordenar los valores de cada variable
    valores_x = np.sort(df[var1])
    valores_y = np.sort(df[var2])


    df_valores_ordenados = pd.DataFrame({
        f'{var1}_ordenado': valores_x,
        f'{var2}_ordenado': valores_y
    })
    matriz = np.zeros((n, n))   # Crear una matriz de ceros donde se anexarán los valores calculados de la cópula

    for i, x_val in enumerate(valores_x):
        for j, y_val in enumerate(valores_y):
            count = np.sum((df[var1] <= x_val) & (df[var2] <= y_val))
            matriz[i, j] = count / n

    return df_valores_ordenados, matriz

df_valores_ordenados, copula= copula_empirica(df, 'V_m3', 'Q_mm') ## Llamamos a la función
df_valores_ordenados.head(10)

,V_m3_ordenado,Q_mm_ordenado
0,4419,121
1,4992,122
2,5002,141
3,5288,168
4,5499,173
5,5657,176
6,5768,181
7,5906,184
8,6414,186
9,6460,187


In [ ]:
copula

array([[0.        , 0.        , 0.        , ..., 0.01851852, 0.01851852,
        0.01851852],
       [0.        , 0.        , 0.        , ..., 0.03703704, 0.03703704,
        0.03703704],
       [0.        , 0.        , 0.        , ..., 0.05555556, 0.05555556,
        0.05555556],
       ...,
       [0.01851852, 0.03703704, 0.05555556, ..., 0.92592593, 0.94444444,
        0.96296296],
       [0.01851852, 0.03703704, 0.05555556, ..., 0.94444444, 0.96296296,
        0.98148148],
       [0.01851852, 0.03703704, 0.05555556, ..., 0.96296296, 0.98148148,
        1.        ]])

In [ ]:
import plotly.graph_objects as go
V_mesh, Q_mesh = np.meshgrid(df_valores_ordenados['V_m3_ordenado'],
                             df_valores_ordenados['Q_mm_ordenado'])
fig = go.Figure(data=[go.Surface(
    x=V_mesh,
    y=Q_mesh,
    z=copula,
    colorscale='Plasma',
    opacity=0.85,
    contours=dict(
        z=dict(show=True, usecolormap=True, highlightcolor="limegreen", project_z=True)
    ),
    hovertemplate='<b>Volumen:</b> %{x:.0f} m³<br><b>Caudal:</b> %{y:.0f} mm<br><b>C(V,Q):</b> %{z:.3f}<extra></extra>'
)])

fig.update_layout(
    title=dict(
        text='Cópula Empírica',
        x=0.5,
        font=dict(size=16)
    ),
    scene=dict(
        xaxis=dict(title='Volumen (m³)', title_font=dict(size=12)),
        yaxis=dict(title='Caudal (m³/s)', title_font=dict(size=12)),
        zaxis=dict(title='C(V,Q)', title_font=dict(size=12)),
        camera=dict(
            eye=dict(x=1.8, y=1.8, z=1.2)
        )
    ),
    width=900,
    height=700
)
fig.show()

### Activos financieros

In [ ]:
###¿Para qué ayuda la cópula empírica en la modelación de dependencia?
import yfinance as yf
activo_1='MSFT'
activo_2='NVDA'

mft=yf.download(activo_1, period="1y" ,interval='1d', auto_adjust=False)
nvidia=yf.download(activo_2, period="1y" ,interval='1d', auto_adjust=False)
datos_empresas=pd.DataFrame()
datos_empresas[activo_1]=mft['Close']
datos_empresas[activo_2]=nvidia['Close']
datos_empresas

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


,MSFT,NVDA
Date,,
2024-10-15,418.739990,131.600006
2024-10-16,416.119995,135.720001
2024-10-17,416.720001,136.929993
2024-10-18,418.160004,138.000000
2024-10-21,418.779999,143.710007
...,...,...
2025-10-08,524.849976,189.110001
2025-10-09,522.400024,192.570007
2025-10-10,510.959991,183.160004


In [ ]:
df_activos, copula_activos= copula_empirica(datos_empresas, activo_1, activo_2) ## Llamamos a la función
df_activos.head(10)

,MSFT_ordenado,NVDA_ordenado
0,354.559998,94.309998
1,357.859985,96.300003
2,359.119995,96.910004
3,359.839996,97.639999
4,366.820007,98.889999
5,367.779999,101.489998
6,371.609985,101.800003
7,373.109985,102.709999
8,374.390015,104.489998
9,375.390015,106.430000


In [ ]:
import plotly.graph_objects as go
X_mesh, Y_mesh = np.meshgrid(df_activos['MSFT_ordenado'],
                             df_activos['NVDA_ordenado'])
fig = go.Figure(data=[go.Surface(
    x=X_mesh,
    y=Y_mesh,
    z=copula_activos,
    colorscale='Plasma',
    opacity=0.85,
    contours=dict(
        z=dict(show=True, usecolormap=True, highlightcolor="limegreen", project_z=True)
    ),
    hovertemplate=f'<b>{activo_1}:</b> %{{x:.2f}}<br><b>{activo_2}:</b> %{{y:.2f}}<br><b>C({activo_1},{activo_2}):</b> %{{z:.3f}}<extra></extra>'
)])

fig.update_layout(
    title=dict(
        text='Cópula Empírica (MSFT vs NVDA)',
        x=0.5,
        font=dict(size=16)
    ),
    scene=dict(
        xaxis=dict(title=f'{activo_1}', title_font=dict(size=12)),
        yaxis=dict(title=f'{activo_2}', title_font=dict(size=12)),
        zaxis=dict(title='C(V,Q)', title_font=dict(size=12)),
        camera=dict(
            eye=dict(x=1.8, y=1.8, z=1.2)
        )
    ),
    width=900,
    height=700
)
fig.show()